In [ ]:
import hashlib
import time
from pathlib import Path

import anthropic
import pandas as pd
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split

data_dir = Path.cwd().parents[1] / 'data'
data_dir.mkdir(exist_ok=True)
load_dotenv(dotenv_path=data_dir.parent / ".env")

client = anthropic.Anthropic()

SUBSAMPLE_SIZE = 900
RANDOM_STATE = 42
SMALL_MODEL = "claude-haiku-4-5"
LARGE_MODEL = "claude-sonnet-5"
MAX_TOKENS_RESPONSE = 600

def request_kwargs(model_name):
    # Sonnet 5 runs adaptive thinking by default (thinking + text share max_tokens),
    # which can exhaust the budget before any answer text is written. Haiku 4.5
    # doesn't support an explicit "disabled" thinking value, so only constrain
    # the model that actually needs it.
    if model_name == LARGE_MODEL:
        return {"thinking": {"type": "disabled"}}
    return {}

In [ ]:
df = pd.read_parquet(data_dir / 'raw_queries_parquet')

In [ ]:
# Step 2a: stratified subsample (3000 -> 900), stable query_id for joining across steps
df["query_id"] = df["query"].apply(lambda q: hashlib.sha1(q.encode()).hexdigest()[:12])

df_sub, _ = train_test_split(
    df,
    train_size=SUBSAMPLE_SIZE,
    stratify=df["source_category"],
    random_state=RANDOM_STATE,
)
df_sub = df_sub.reset_index(drop=True)
df_sub.to_parquet(data_dir / "subsampled_queries.parquet", index=False)

print(f"Subsampled {len(df_sub)} queries across {df_sub['source_category'].nunique()} categories")
print(df_sub["source_category"].value_counts())
print(f"Unique query_id count: {df_sub['query_id'].nunique()} (should equal {len(df_sub)})")

In [ ]:
# Step 2 smoke-test helper: fixed to distinguish real failures from real answers
def run_single_query(query, model_name):
    try:
        response = client.messages.create(
            model=model_name,
            max_tokens=MAX_TOKENS_RESPONSE,
            messages=[{"role": "user", "content": query}],
            **request_kwargs(model_name),
        )
        text = next((b.text for b in response.content if b.type == "text"), None)
        return {"output": text, "error": None}
    except (anthropic.RateLimitError, anthropic.APIStatusError, anthropic.APIConnectionError) as e:
        return {"output": None, "error": str(e)}

In [ ]:
# Smoke test: confirm both models respond before committing to the full 1,800-request batch
smoke_sample = df_sub.head(5)
for row in smoke_sample.itertuples():
    small_result = run_single_query(row.query, SMALL_MODEL)
    large_result = run_single_query(row.query, LARGE_MODEL)
    print(f"--- {row.query_id} ---")
    print(f"query: {row.query[:80]!r}")
    print(f"small ({SMALL_MODEL}): {small_result}")
    print(f"large ({LARGE_MODEL}): {large_result}")

In [ ]:
# Step 2b: build and submit the full batch (900 queries x 2 models = 1,800 requests)
def build_model_batch_requests(df):
    requests = []
    for row in df.itertuples():
        for slot, model in (("small", SMALL_MODEL), ("large", LARGE_MODEL)):
            requests.append(
                Request(
                    custom_id=f"{row.query_id}__{slot}",
                    params=MessageCreateParamsNonStreaming(
                        model=model,
                        max_tokens=MAX_TOKENS_RESPONSE,
                        messages=[{"role": "user", "content": row.query}],
                        **request_kwargs(model),
                    ),
                )
            )
    return requests

batch = client.messages.batches.create(requests=build_model_batch_requests(df_sub))
print(f"Submitted batch {batch.id}, status={batch.processing_status}")

In [ ]:
# Poll until the batch finishes (usually well under an hour; max 24h)
def wait_for_batch(batch_id, poll_seconds=60):
    while True:
        b = client.messages.batches.retrieve(batch_id)
        if b.processing_status == "ended":
            return b
        print(f"status={b.processing_status}, request_counts={b.request_counts}")
        time.sleep(poll_seconds)

batch = wait_for_batch(batch.id)
print(f"Batch ended. request_counts={batch.request_counts}")

In [ ]:
# Parse results: a failure (or "succeeded but no text", e.g. thinking ate the whole
# max_tokens budget) is never written into an output column, it's logged instead
def extract_text(result):
    return next((b.text for b in result.result.message.content if b.type == "text"), None)

def parse_model_batch_results(batch_id, df):
    outputs = {}
    failed_rows = []
    for result in client.messages.batches.results(batch_id):
        query_id, slot = result.custom_id.split("__")
        outputs.setdefault(query_id, {})
        text = extract_text(result) if result.result.type == "succeeded" else None
        outputs[query_id][f"{slot}_model_output"] = text
        if result.result.type != "succeeded":
            failed_rows.append({"query_id": query_id, "slot": slot, "error": result.result.type})
        elif text is None:
            failed_rows.append({"query_id": query_id, "slot": slot, "error": f"empty_text:{result.result.message.stop_reason}"})

    outputs_df = pd.DataFrame.from_dict(outputs, orient="index").reset_index(names="query_id")
    out_df = df.merge(outputs_df, on="query_id", how="left")
    out_df.to_parquet(data_dir / "step2_model_outputs.parquet", index=False)

    failed_path = data_dir / "step2_failed_queries.csv"
    if failed_rows:
        pd.DataFrame(failed_rows).to_csv(failed_path, index=False)
    elif failed_path.exists():
        failed_path.unlink()

    return out_df, failed_rows

out_df, failed = parse_model_batch_results(batch.id, df_sub)
print(f"Saved data/step2_model_outputs.parquet. {len(failed)} failed (query_id, slot) pairs.")

In [ ]:
# Retry failed (query_id, slot) pairs, up to 2 rounds. Re-run this cell to retry again if needed.
def build_retry_requests(df, failed_df):
    query_by_id = df.set_index("query_id")["query"].to_dict()
    model_by_slot = {"small": SMALL_MODEL, "large": LARGE_MODEL}
    requests = []
    for row in failed_df.itertuples():
        model = model_by_slot[row.slot]
        requests.append(
            Request(
                custom_id=f"{row.query_id}__{row.slot}",
                params=MessageCreateParamsNonStreaming(
                    model=model,
                    max_tokens=MAX_TOKENS_RESPONSE,
                    messages=[{"role": "user", "content": query_by_id[row.query_id]}],
                    **request_kwargs(model),
                ),
            )
        )
    return requests

retries = 0
while failed and retries < 2:
    retries += 1
    print(f"Retry round {retries}: resubmitting {len(failed)} failed pairs...")
    retry_batch = client.messages.batches.create(requests=build_retry_requests(df_sub, pd.DataFrame(failed)))
    retry_batch = wait_for_batch(retry_batch.id)

    out_df = pd.read_parquet(data_dir / "step2_model_outputs.parquet")
    still_failed = []
    for result in client.messages.batches.results(retry_batch.id):
        query_id, slot = result.custom_id.split("__")
        col = f"{slot}_model_output"
        text = extract_text(result) if result.result.type == "succeeded" else None
        out_df.loc[out_df["query_id"] == query_id, col] = text
        if result.result.type != "succeeded":
            still_failed.append({"query_id": query_id, "slot": slot, "error": result.result.type})
        elif text is None:
            still_failed.append({"query_id": query_id, "slot": slot, "error": f"empty_text:{result.result.message.stop_reason}"})

    out_df.to_parquet(data_dir / "step2_model_outputs.parquet", index=False)
    failed_path = data_dir / "step2_failed_queries.csv"
    if still_failed:
        pd.DataFrame(still_failed).to_csv(failed_path, index=False)
    elif failed_path.exists():
        failed_path.unlink()

    failed = still_failed
    print(f"After retry {retries}: {len(failed)} still failed.")

print(f"Step 2 done. Final failed count: {len(failed)}.")